# Compute cost of the benchmark runs
How long does a full active-learning run take, and how does the runtime scale with training set size?

Each benchmark run writes a `timing.csv` into its own
`simulation_output/.../train_size_<size>/` folder, recording the wall-clock
`seconds` for that (dataset, representation, seed, training size) run plus the
hardware it ran on. This notebook collects them all and reports **runtime per
training size**, as mean +/- std across the 14 benchmark datasets (10
substitution + 3 AMP + 1 HemoPI2), so you get a single interpretable
cost-vs-training-size table.


In [1]:
import sys, os, glob, re
try:
    import analysis_common as ac
except ModuleNotFoundError:
    sys.path.insert(0, os.getcwd()); import analysis_common as ac
import numpy as np, pandas as pd

BENCH_OUT = ac.BENCH_OUT
print('benchmark output root:', BENCH_OUT)

# Collect every timing.csv under the benchmark output tree (both the baseline
# sweep <family>/<rep>/... and the foundation sweep foundations/<family>/<model>/...).
paths = glob.glob(os.path.join(BENCH_OUT, '**', 'timing.csv'), recursive=True)
print(f'found {len(paths)} timing.csv files')

frames = []
for p in paths:
    try:
        t = pd.read_csv(p)
    except Exception as e:
        print('  [skip]', p, e); continue
    if t.empty:
        continue
    # recover whether this came from the foundation sweep, from the path
    t['from_foundations'] = (os.sep + 'foundations' + os.sep) in p
    frames.append(t)

if not frames:
    raise SystemExit('No timing data found. Run the benchmark first (it writes '
                     'timing.csv into each run folder).')
tim = pd.concat(frames, ignore_index=True)
# normalise types
tim['Train Size'] = pd.to_numeric(tim['Train Size'], errors='coerce')
tim['seconds'] = pd.to_numeric(tim['seconds'], errors='coerce')
tim = tim.dropna(subset=['Train Size', 'seconds'])
print('rows:', len(tim),
      '| datasets:', tim['Dataset'].nunique(),
      '| representations:', sorted(tim['Representation'].dropna().unique()),
      '| train sizes:', sorted(tim['Train Size'].unique().astype(int)))


benchmark output root: /Users/leoan/Documents/GitHub/SCARSE-repro/code/benchmark/simulate_experiments/simulation_output
found 11040 timing.csv files
rows: 11040 | datasets: 24 | representations: ['esm', 'morgan', 'ngram', 'pepbert', 'physchem'] | train sizes: [np.int64(20), np.int64(40), np.int64(50), np.int64(60), np.int64(75), np.int64(80), np.int64(100), np.int64(200), np.int64(350), np.int64(500)]


## Hardware the timings were measured on
Timings are only interpretable alongside the machine. If several machines were
used (e.g. GPU vs CPU nodes), they are listed separately.


In [2]:
hw_cols = [c for c in ['device','gpu_name','gpu_mem_gb','cpu_model','cpu_count_physical',
                       'cpu_count_logical','ram_total_gb','cuda_version','torch_version',
                       'slurm_cpus_per_task','omp_num_threads'] if c in tim.columns]
if hw_cols:
    hw = (tim[hw_cols].astype(str)
          .value_counts().rename('runs').reset_index())
    # show the distinct hardware profiles, most-used first
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print(hw.to_string(index=False))
else:
    print('No hardware columns present in timing.csv.')


device gpu_name gpu_mem_gb                       cpu_model cpu_count_physical cpu_count_logical ram_total_gb cuda_version torch_version slurm_cpus_per_task omp_num_threads  runs
   cpu     none       none AMD EPYC 9534 64-Core Processor                128               256       1217.1         none   2.9.0+cu128                   4               4 11040


## Runtime per training size (mean +/- std across the 14 datasets)
This is the headline table. For a chosen representation, each row is one
training size; `mean_s` / `std_s` are the mean and standard deviation of the
per-run wall-clock time across the datasets (and seeds), so it reads as
"a run at this training size takes X +/- Y seconds".


In [3]:
# Choose which representation to report. SCARSE = 'esm' is the model of record;
# set REP = None to pool all representations, or e.g. 'pepbert'.
REP = 'esm'

def runtime_table(df, rep=REP, per='run'):
    d = df if rep is None else df[df['Representation'] == rep]
    if d.empty:
        print(f'(no timing rows for representation {rep!r})'); return None
    # one row per (dataset, seed, train size) is already the unit; aggregate
    # across datasets AND seeds at each training size.
    g = (d.groupby('Train Size')['seconds']
         .agg(n='count', mean_s='mean', std_s='std', min_s='min', max_s='max',
              median_s='median')
         .reset_index().sort_values('Train Size'))
    g['std_s'] = g['std_s'].fillna(0.0)
    # also give minutes for the larger sizes
    g['mean_min'] = (g['mean_s'] / 60).round(2)
    g['std_min'] = (g['std_s'] / 60).round(2)
    for c in ['mean_s','std_s','min_s','max_s','median_s']:
        g[c] = g[c].round(2)
    g['Train Size'] = g['Train Size'].astype(int)
    return g

tbl = runtime_table(tim, rep=REP)
if tbl is not None:
    print(f'Runtime per training size  (representation = {REP!r}, '
          f'aggregated across {tim[tim.Representation==REP].Dataset.nunique() if REP else tim.Dataset.nunique()} datasets x seeds)')
    print()
    print(tbl.to_markdown(index=False))


Runtime per training size  (representation = 'esm', aggregated across 24 datasets x seeds)

|   Train Size |   n |   mean_s |   std_s |   min_s |   max_s |   median_s |   mean_min |   std_min |
|-------------:|----:|---------:|--------:|--------:|--------:|-----------:|-----------:|----------:|
|           20 | 960 |    42.55 |   60.91 |    7.34 |  325.3  |      17.32 |       0.71 |      1.02 |
|           40 | 400 |    19.91 |   10.29 |    8.29 |   57.1  |      16.03 |       0.33 |      0.17 |
|           50 | 560 |    70.25 |   78.95 |    9.32 |  391.41 |      30.53 |       1.17 |      1.32 |
|           60 | 400 |    24.94 |   14.14 |    9.34 |   67.32 |      19.77 |       0.42 |      0.24 |
|           75 | 560 |    78.9  |   86.75 |   11.22 |  434.06 |      36.97 |       1.31 |      1.45 |
|           80 | 400 |    30.69 |   18.88 |    9.86 |  107.11 |      23.52 |       0.51 |      0.31 |
|          100 | 560 |    92.31 |  100.09 |   12.84 |  538.65 |      43.95 |       1.54 |   

## Same table, split by dataset family
So you can see whether AMP vs substitution vs HemoPI2 runs cost differently at a
given training size (e.g. AMP pools are larger, so embedding more sequences).


In [4]:
def family_of(name):
    # friendly ids and benchmark CSV basenames both handled
    amp = {'Ecoli_AMPs','Saureus_AMPs','Paeruginosa_AMPs','EC_all_40','SA_all_40','PA_all_40'}
    hemo = {'HemoPI2','hemopi2_all'}
    if name in amp: return 'AMP'
    if name in hemo: return 'HemoPI2'
    if '_indels' in str(name) or str(name).endswith('indels'): return 'Indel'
    return 'Substitution'

d = tim if REP is None else tim[tim['Representation'] == REP]
d = d.copy(); d['family'] = d['Dataset'].map(family_of)
fam_tbl = (d.groupby(['family','Train Size'])['seconds']
           .agg(n='count', mean_s='mean', std_s='std').reset_index())
fam_tbl['std_s'] = fam_tbl['std_s'].fillna(0.0)
fam_tbl[['mean_s','std_s']] = fam_tbl[['mean_s','std_s']].round(2)
fam_tbl['Train Size'] = fam_tbl['Train Size'].astype(int)
fam_tbl = fam_tbl.sort_values(['family','Train Size'])
print(fam_tbl.to_markdown(index=False))


| family       |   Train Size |   n |   mean_s |   std_s |
|:-------------|-------------:|----:|---------:|--------:|
| AMP          |           20 | 120 |    32.04 |   28.41 |
| AMP          |           50 | 120 |    40.23 |   35.96 |
| AMP          |           75 | 120 |    45.26 |   36.81 |
| AMP          |          100 | 120 |    54.35 |   44.63 |
| AMP          |          200 | 120 |    98.46 |   69.41 |
| AMP          |          350 | 120 |   224.49 |  140.22 |
| AMP          |          500 | 120 |   417.29 |  223.83 |
| HemoPI2      |           20 |  40 |    32.38 |   30.82 |
| HemoPI2      |           50 |  40 |    40.42 |   35.26 |
| HemoPI2      |           75 |  40 |    44.96 |   35.69 |
| HemoPI2      |          100 |  40 |    52.87 |   42.49 |
| HemoPI2      |          200 |  40 |    98.01 |   64.76 |
| HemoPI2      |          350 |  40 |   227.35 |  121.04 |
| HemoPI2      |          500 |  40 |   432.96 |  219.44 |
| Indel        |           20 | 400 |    15.63 |    6.7 

## Optional: compare representations at each training size
A quick cross-tab of mean seconds per representation x training size, so the
cost of SCARSE (ESM-2) vs the cheaper baselines is visible at a glance.


In [5]:
pivot = (tim.groupby(['Representation','Train Size'])['seconds'].mean()
         .round(2).unstack('Train Size'))
print('Mean seconds per run  (rows = representation, cols = training size):')
print(pivot.to_markdown())

# Save the headline table to CSV for the manuscript / SI.
if 'tbl' in dir() and tbl is not None:
    out = os.path.join(ac.FIG_DIR, 'runtime_per_training_size.csv')
    tbl.to_csv(out, index=False); print('\nsaved', out)


Mean seconds per run  (rows = representation, cols = training size):
| Representation   |    20 |    40 |    50 |    60 |    75 |    80 |   100 |    200 |    350 |    500 |
|:-----------------|------:|------:|------:|------:|------:|------:|------:|-------:|-------:|-------:|
| esm              | 42.55 | 19.91 | 70.25 | 24.94 | 78.9  | 30.69 | 92.31 | 152.78 | 305.25 | 528.4  |
| morgan           |  4.22 |  6.26 |  9.66 |  9.93 | 16.86 | 15.27 | 25.98 |  90.67 | 252.37 | 467.44 |
| ngram            |  3.89 |  3.44 |  7.93 |  4.54 | 14.21 |  6.71 | 24.44 |  94.37 | 357.84 | 893.75 |
| pepbert          |  8.75 |  8.96 | 12.89 | 11.49 | 15.76 | 14.18 | 20.05 |  45.23 | 117.5  | 217.19 |
| physchem         |  3.69 |  4.8  |  4.81 |  5.79 |  5.6  |  7.39 |  6.67 |  25.41 |  92.88 | 194.61 |

saved /Users/leoan/Documents/GitHub/SCARSE-repro/code/analysis/figures/runtime_per_training_size.csv
